# Week 4 A: Practical LLMs, VLMs and Agent Workflows
Open this notebook in Course_Sessions/Week_4/activities/Session_A. Keep llm_lab.py and lab_data beside it. Use the mlcourse kernel.

Read README.md before live calls. Install/start Ollama separately or use the lecturer's configured endpoint. Model downloads are explicit setup steps. The helper never downloads models.

For each cell: predict, match the comments to the numbered blocks, run, inspect and explain. Use your own notebook for saved work. These are class activities, not an assessment brief. All profiles, invoice values, drawing dimensions and model-results rows are synthetic. Research records are source-linked teaching summaries, not full-paper extracts.


## Open the Week 4 activity folder

Open Course_Sessions/Week_4/activities/Session_A. Create your own notebook there and choose the mlcourse kernel.

Match each comment to its block:

- A: Inspect the supplied files.
- B: Import notebook utilities.
- C: Check the working folder and import supplied helpers.

Check after running: You should see course_sources.json, candidate.json, research_extracts.json and invoice_reference.json.


In [ ]:
# Block 1: add the matching comment
import json
from pathlib import Path
from IPython.display import display, Image

# Block 2: add the matching comment
if not Path("llm_lab.py").exists():
    raise FileNotFoundError("Open the notebook in Course_Sessions/Week_4/activities/Session_A.")
from llm_lab import DATA, load_json, chat, json_answer, backend_status

# Block 3: add the matching comment
print("Data folder:", DATA.name)
print("Teaching files:", sorted(p.name for p in DATA.glob("*.json")))


## Check the model service

Start the local service or use the endpoint supplied by the lecturer.

Match each comment to its block:

- A: Request the list of installed models.
- B: Inspect available models before live activities.
- C: Import the model-service check.

Check after running: Match the text and vision model names with the available list. A connection error means the service needs setup.


In [ ]:
# Block 1: add the matching comment
from llm_lab import backend_status

# Block 2: add the matching comment
status = backend_status()

# Block 3: add the matching comment
print(json.dumps(status, indent=2))


## Make a real language-model request

Ask a familiar question, then change the instruction and compare the answer.

Match each comment to its block:

- A: Prepare a question and import the request helper.
- B: Read the answer and measured request metadata.
- C: Send the request to the real model service.

Check after running: Change the question or sentence limit. Judge correctness separately from fluency and token count.


In [ ]:
# Block 1: add the matching comment
from llm_lab import chat
question = "Explain classification to a beginner using a clothing example."

# Block 2: add the matching comment
response = chat(question, system="Use three short sentences.", temperature=0.2)

# Block 3: add the matching comment
print(response["text"])
print("Model:", response["model"])
print("Seconds:", response["seconds"], "Output tokens:", response["output_tokens"])


## Extract a structured request

Turn a short message into fields that ordinary code can inspect.

Match each comment to its block:

- A: Inspect the parsed fields.
- B: Request schema-constrained output.
- C: Describe the message and required JSON fields.

Check after running: Valid fields establish the format. Read the original message to check the extracted meaning.


In [ ]:
# Block 1: add the matching comment
message = "Compare Random Forest and logistic regression using accuracy."
schema = {"type": "object", "properties": {
    "models": {"type": "array", "items": {"type": "string"}},
    "metric": {"type": "string"}},
    "required": ["models", "metric"], "additionalProperties": False}

# Block 2: add the matching comment
request = json_answer("Extract the model names into models and the metric name into metric: " + message, schema)

# Block 3: add the matching comment
print(json.dumps(request, indent=2))
print("Requested metric:", request["metric"])


## Retrieve course passages without a model service

Run a TF-IDF baseline and inspect what it retrieves.

Match each comment to its block:

- A: Retrieve three passages with TF-IDF cosine similarity.
- B: Prepare a question and the supplied retrieval helper.
- C: Read each result and its source identifier.

Check after running: The scaling passage should be retrieved. TF-IDF uses word statistics; this cell does not run an embedding model.


In [ ]:
# Block 1: add the matching comment
from llm_lab import search_sources
question = "Why fit the scaler using only training features?"

# Block 2: add the matching comment
passages = search_sources(question, k=3)

# Block 3: add the matching comment
for passage in passages:
    print(passage["id"], passage["title"], passage["score"])
    print(passage["text"], "\n")


## Answer using the retrieved evidence

Make a real model call, then check every cited passage.

Match each comment to its block:

- A: Prepare the question and retrieve its context.
- B: Generate an answer using only those passages.
- C: Check citation identifiers and read their supporting text.

Check after running: An answer should explain leakage. A valid citation ID still needs a factual-support check.


In [ ]:
# Block 1: add the matching comment
from llm_lab import search_sources, rag_answer
question = "Why should the scaler be fitted on training features only?"
passages = search_sources(question, k=3)

# Block 2: add the matching comment
answer = rag_answer(question, passages)

# Block 3: add the matching comment
print(json.dumps(answer, indent=2))
print("Source IDs:", [p["id"] for p in passages])


## Compare a learned embedding search

Compare the TF-IDF baseline with actual embedding-model inference.

Match each comment to its block:

- A: Compare relevant passages rather than treating scores as probabilities.
- B: Prepare a paraphrased question.
- C: Run both retrieval methods on the same corpus.

Check after running: This cell needs the all-minilm model. Record which retrieval method finds useful evidence for your question.


In [ ]:
# Block 1: add the matching comment
from llm_lab import search_sources, semantic_search
question = "How can held-out information accidentally affect preprocessing?"

# Block 2: add the matching comment
baseline = search_sources(question, k=3)
semantic = semantic_search(question, k=3)

# Block 3: add the matching comment
print("TF-IDF:", [(p["id"], p["score"]) for p in baseline])
print("Learned embeddings:", [(p["id"], p["score"]) for p in semantic])


## Inspect the input before asking the VLM

Read the same synthetic image that the vision model will receive.

Match each comment to its block:

- A: Display the actual image supplied to the model.
- B: Inspect the known reference for later evaluation.
- C: Locate the image and independent reference values.

Check after running: The image contains INV-042, two notebooks, three pens and a printed total of NZD 35.65.


In [ ]:
# Block 1: add the matching comment
invoice_image = DATA / "invoice.png"
reference = load_json("invoice_reference.json")

# Block 2: add the matching comment
display(Image(filename=str(invoice_image)))

# Block 3: add the matching comment
print("Reference total:", reference["total"], reference["currency"])


## Extract the invoice with a real VLM

Request structured fields from pixels, then inspect the result.

Match each comment to its block:

- A: Use the supplied invoice schema.
- B: Inspect the extracted values against the displayed image.
- C: Send the image and question to the vision model.

Check after running: The vision model is separate from the text model. Report extraction errors rather than replacing them silently.


In [ ]:
# Block 1: add the matching comment
from llm_lab import INVOICE_SCHEMA, json_answer

# Block 2: add the matching comment
invoice = json_answer(
    "Read the synthetic invoice. Extract the printed fields exactly.",
    INVOICE_SCHEMA, images=[DATA / "invoice.png"])

# Block 3: add the matching comment
print(json.dumps(invoice, indent=2))


## Check arithmetic and source agreement

Use ordinary Python for exact calculations after extraction.

Match each comment to its block:

- A: Compare the extracted total with an independent source reference.
- B: Calculate line totals and check internal agreement.
- C: Import the arithmetic verifier.

Check after running: Two internally consistent wrong numbers can pass arithmetic. Check quantities, prices and all source fields too.


In [ ]:
# Block 1: add the matching comment
from llm_lab import check_invoice

# Block 2: add the matching comment
checks = check_invoice(invoice)
print(json.dumps(checks, indent=2))

# Block 3: add the matching comment
reference = load_json("invoice_reference.json")
print("Extracted total:", invoice["total"])
print("Reference total:", reference["total"])
print("Total matches source:", abs(invoice["total"] - reference["total"]) < 0.005)


## Compare two drawing revisions

Extract the printed heights and calculate the change.

Match each comment to its block:

- A: Ask the VLM to read the printed dimensions.
- B: Display both source revisions and define the expected fields.
- C: Calculate the difference using ordinary code.

Check after running: The reference dimensions are 2850 and 3000 mm. The difference is 150 mm; the exercise reads labels rather than measuring pixels.


In [ ]:
# Block 1: add the matching comment
from llm_lab import compare_height
display(Image(filename=str(DATA / "plan_A.png")))
display(Image(filename=str(DATA / "plan_B.png")))
schema = {"type": "object", "properties": {
    "height_A_mm": {"type": "number"}, "height_B_mm": {"type": "number"}},
    "required": ["height_A_mm", "height_B_mm"], "additionalProperties": False}

# Block 2: add the matching comment
heights = json_answer("Read the printed wall W17 height in A and B.",
    schema, images=[DATA / "plan_A.png", DATA / "plan_B.png"])

# Block 3: add the matching comment
print(heights)
print(compare_height(heights["height_A_mm"], heights["height_B_mm"]))


## Inspect one fair model comparison

Read synthetic results and exclude the row from a different test split.

Match each comment to its block:

- A: Import the table and comparison tools.
- B: Display all synthetic teaching results.
- C: Compare rows from one split and inspect exclusions.

Check after running: Dense Network has the largest accuracy on test_A. The test_B row cannot establish a fair win against it.


In [ ]:
# Block 1: add the matching comment
import pandas as pd
from llm_lab import model_results

# Block 2: add the matching comment
table = pd.read_csv(DATA / "model_results.csv")
display(table)

# Block 3: add the matching comment
comparison = model_results("test_A")
print(json.dumps(comparison, indent=2))


## Let the model choose a results tool

Inspect every tool call before accepting the final explanation.

Match each comment to its block:

- A: Inspect the tool evidence and final claims.
- B: Prepare a task for the bounded agent.
- C: Run real model-selected tools under a turn limit.

Check after running: Look for inspect_results with split test_A. Explain a wrong argument or an unsupported final claim if one occurs.


In [ ]:
# Block 1: add the matching comment
from llm_lab import run_agent
goal = "Compare accuracy on test_A. Explain why the test_B row is excluded."

# Block 2: add the matching comment
run = run_agent(goal, case="data", max_turns=5)

# Block 3: add the matching comment
print("Status:", run["status"])
display(run["trace"])
print(run["answer"])


## Run a fixed review workflow

Follow a predefined extraction, calculation and review sequence.

Match each comment to its block:

- A: Run extraction and arithmetic in a fixed order.
- B: Route the result using a named rule and retain source review.
- C: Use the same supplied schema and verifier.

Check after running: The application chooses this order. Arithmetic passed is a narrow check, not an approval of the document.


In [ ]:
# Block 1: add the matching comment
from llm_lab import check_invoice, INVOICE_SCHEMA

# Block 2: add the matching comment
record = json_answer("Read the printed invoice fields.", INVOICE_SCHEMA,
                     images=[DATA / "invoice.png"])
checks = check_invoice(record)

# Block 3: add the matching comment
outcome = "arithmetic_passed" if all([
    checks["subtotal_agrees"], checks["total_agrees"]]) else "review_required"
print(outcome)
print("Next: compare every field with the image, then record a human decision.")


## Build a requirement-to-evidence table

Use a fictional profile and retrieve support before drafting.

Match each comment to its block:

- A: Load the supplied profile and fictional advertisement.
- B: Read the role requirements alongside the evidence.
- C: Retrieve evidence for each named requirement.

Check after running: There is no Kubernetes evidence in this profile. Record that gap instead of inventing experience.


In [ ]:
# Block 1: add the matching comment
from llm_lab import search_sources
candidate = load_json("candidate.json")
job = (DATA / "job_ad.txt").read_text()

# Block 2: add the matching comment
for requirement in ["Python", "machine learning", "communication", "Kubernetes"]:
    matches = search_sources(requirement, candidate["records"], k=2)
    print(requirement, [(r["id"], r["evidence"]) for r in matches])

# Block 3: add the matching comment
print(job)


## Draft claims with exact evidence quotations

Require a source ID and quotation for each application claim.

Match each comment to its block:

- A: Read the draft claims with their quotations.
- B: Generate claims from the supplied profile only.
- C: Define the claim and evidence fields.

Check after running: The applicant is fictional. Check that each claim is no stronger than its supporting record.


In [ ]:
# Block 1: add the matching comment
claim_schema = {"type": "object", "properties": {
    "claims": {"type": "array", "items": {"type": "object", "properties": {
        "claim": {"type": "string"}, "source_id": {"type": "string"},
        "quote": {"type": "string"}}, "required": ["claim", "source_id", "quote"],
        "additionalProperties": False}}}, "required": ["claims"], "additionalProperties": False}

# Block 2: add the matching comment
draft = json_answer("Prepare three application claims using only these records. "
    "Copy the supporting quote exactly.\n" + json.dumps(candidate), claim_schema)

# Block 3: add the matching comment
display(draft["claims"])


## Check the evidence before revision

Inspect exact quotations, then judge whether they support the claim.

Match each comment to its block:

- A: Check source identifiers and exact quotations.
- B: Import the deterministic evidence checks.
- C: Request a separate meaning review against the same evidence.

Check after running: A matching quote can still support only part of a claim. Explain one acceptance or revision in your own words.


In [ ]:
# Block 1: add the matching comment
from llm_lab import check_claims

# Block 2: add the matching comment
checked = check_claims(draft["claims"], candidate["records"])
display(checked)

# Block 3: add the matching comment
revision = chat("Review these application claims. Identify unsupported wording. "
    "Use only the evidence records.\n" + json.dumps({
        "claims": checked, "evidence": candidate["records"]}), temperature=0)
print(revision["text"])


## Inspect supplied research summaries

Distinguish the supplied summaries from full-paper evidence.

Match each comment to its block:

- A: Load three source-linked teaching summaries.
- B: Inspect their provenance and available evidence.
- C: State the limitation before making a comparison.

Check after running: Do not invent dataset scores or claim to have examined full papers from this file.


In [ ]:
# Block 1: add the matching comment
from llm_lab import CLAIM_SCHEMA, check_claims
papers = load_json("research_extracts.json")

# Block 2: add the matching comment
for paper in papers:
    print(paper["id"], paper["title"])
    print(paper["kind"])
    print(paper["text"], "\n")

# Block 3: add the matching comment
print("Shared benchmark scores are not included in these summaries.")


## Prepare a source-supported comparison

Extract claims that the supplied summaries actually support.

Match each comment to its block:

- A: Check identifiers and quotations before reading for factual support.
- B: Prepare a scoped research question and its source data.
- C: Extract structured claims with evidence.

Check after running: Explain why different research tasks and missing shared benchmarks prevent an accuracy ranking here.


In [ ]:
# Block 1: add the matching comment
paper_prompt = "Compare the three research tasks. Give three claims with "
paper_prompt += "source_id and an exact quote from the supplied summaries. "
paper_prompt += "Do not rank the papers by accuracy.\n" + json.dumps(papers)

# Block 2: add the matching comment
research = json_answer(paper_prompt, CLAIM_SCHEMA)

# Block 3: add the matching comment
display(check_claims(research["claims"], papers))


## Observe the practice bug

Check a small arithmetic function with known examples.

Match each comment to its block:

- A: Inspect the code and run the bounded arithmetic checker.
- B: Compare actual values with the four expected results.
- C: Read the supplied practice function.

Check after running: The original denominator adds one. Only the zero-correct case passes the supplied checks.


In [ ]:
# Block 1: add the matching comment
from llm_lab import test_candidate
buggy = (DATA / "practice_bug.py.txt").read_text().split("\n\nprint")[0]

# Block 2: add the matching comment
print(buggy)
baseline = test_candidate(buggy)

# Block 3: add the matching comment
display(baseline)


## Request and check a candidate correction

Inspect generated code without executing arbitrary model output.

Match each comment to its block:

- A: Define the expected code-response field.
- B: Interpret the small allowed arithmetic syntax and inspect test outcomes.
- C: Ask the real model for a candidate correction.

Check after running: The checker rejects imports and arbitrary statements. Passing four examples is evidence for those examples.


In [ ]:
# Block 1: add the matching comment
fix_schema = {"type": "object", "properties": {"code": {"type": "string"}},
              "required": ["code"], "additionalProperties": False}

# Block 2: add the matching comment
proposal = json_answer("Fix the supplied accuracy function. Return only one "
    "function named accuracy(correct, total), with one arithmetic return.\n" + buggy,
    fix_schema)

# Block 3: add the matching comment
print(proposal["code"])
display(test_candidate(proposal["code"]))


## Inspect a debugging agent trace

Let the agent read and test using the same bounded tools.

Match each comment to its block:

- A: Inspect the actual evidence behind the final explanation.
- B: Run real model-selected read and test tools.
- C: Define a small debugging task.

Check after running: A finished agent must still be checked for a test_candidate result that passed. Inspect cases where it finishes too early.


In [ ]:
# Block 1: add the matching comment
from llm_lab import run_agent
goal = "Read the practice accuracy function. Correct the denominator and test it."

# Block 2: add the matching comment
debugging = run_agent(goal, case="coding", max_turns=5)

# Block 3: add the matching comment
display(debugging["trace"])
print(debugging["status"], debugging["answer"])


## Run a small assistant across tasks

Compare the tool path for a data question, a course question and an invoice check.

Match each comment to its block:

- A: Run each task with the same bounded tool registry.
- B: Prepare three distinct tasks for the same assistant.
- C: Compare selected tools and inspect task evidence.

Check after running: The invoice tool checks the supplied JSON reference; image extraction was the separate VLM activity.


In [ ]:
# Block 1: add the matching comment
from llm_lab import run_agent
requests = ["Compare accuracy on test_A.",
    "Find course evidence explaining why a scaler is fitted on training data.",
    "Check the arithmetic of the supplied invoice reference."]

# Block 2: add the matching comment
reports = [run_agent(task, case="capstone", max_turns=5) for task in requests]

# Block 3: add the matching comment
for task, report in zip(requests, reports):
    print(task, report["status"])
    print([step["tool"] for step in report["trace"]])
    print(report["answer"], "\n")


## Record outcomes separately from completion

Create an evidence table and add your own factual checks.

Match each comment to its block:

- A: Import the table utility.
- B: Record observed execution outcomes and an explicit pending correctness check.
- C: Inspect the table and complete your own evidence review.

Check after running: Add a short explanation of one successful result and one failure or unresolved result. Completion alone is not correctness.


In [ ]:
# Block 1: add the matching comment
import pandas as pd

# Block 2: add the matching comment
summary = pd.DataFrame([{
    "task": task, "status": report["status"],
    "tool_calls": len(report["trace"]),
    "tools": ", ".join(step["tool"] for step in report["trace"]),
    "human_correctness_check": "Not yet reviewed"
} for task, report in zip(requests, reports)])

# Block 3: add the matching comment
display(summary)
